# OpenVLA LIBERO Learning Lab

这个 notebook 是给你快速接手 `openvla_line/` 用的。它的重点不是花哨可视化，而是把下面这条链讲清楚并能自己动手验证：

1. `LIBERO hdf5` 里到底有什么
2. manifest 是怎么切出来的
3. 单个 sample 的输入输出长什么样
4. 模型 forward 的张量形状是什么
5. 正式训练该怎么启动


In [ ]:
from pathlib import Path
import json
import sys

candidate_roots = [
    Path.cwd().resolve(),
    Path('/home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/openvla_line'),
]
PROJECT_ROOT = next(root for root in candidate_roots if (root / 'openvla_line').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT.parent / 'libero' / 'headless_tools' / 'data' / 'libero_datasets' / 'libero_spatial'
RUN_ROOT = PROJECT_ROOT / 'artifacts' / 'runs'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('data_exists =', DATA_ROOT.exists())
print('num_hdf5 =', len(list(DATA_ROOT.glob('*.hdf5'))) if DATA_ROOT.exists() else 0)

## 1. 先看真实 LIBERO 数据长什么样

当前这条线默认吃的是：

- `agentview_rgb`
- `eye_in_hand_rgb`
- `robot_states`
- 任务语言
- `actions`

下面这格会直接对着一份真实 demo 做 manifest 级别的读取。

In [ ]:
from openvla_line.config import load_experiment_config, resolve_suite_dir
from openvla_line.data import build_libero_manifest

config = load_experiment_config(PROJECT_ROOT / 'configs' / 'libero_spatial_smoke.json')
suite_dir = resolve_suite_dir(config)
manifest = build_libero_manifest(suite_dir=suite_dir, train_ratio=config.data.train_ratio, seed=config.data.seed, max_files=1)

summary = {
    'suite_dir': str(suite_dir),
    'num_demo_files': manifest['num_demo_files'],
    'num_train_episodes': manifest['num_train_episodes'],
    'num_val_episodes': manifest['num_val_episodes'],
    'action_dim': manifest['action_dim'],
    'proprio_dim': manifest['proprio_dim'],
    'available_cameras': manifest['available_cameras'],
    'first_train_episode': manifest['train_episodes'][0],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 2. 看一个 sample 和模型 forward

这格的目的很实用：你能直接看到 dataset 返回的张量形状，以及模型 forward 后输出的动作形状。

In [ ]:
import torch

from openvla_line.data import LiberoStepDataset
from openvla_line.model import OpenVLAStylePolicy
from openvla_line.tokenizer import SimpleTokenizer

texts = [entry['instruction'] for entry in manifest['train_episodes'] + manifest['val_episodes']]
tokenizer = SimpleTokenizer.build_from_texts(texts)

dataset = LiberoStepDataset(
    manifest=manifest,
    suite_dir=suite_dir,
    split='train',
    tokenizer=tokenizer,
    camera_names=config.data.camera_names,
    image_size=config.data.image_size,
    max_text_tokens=config.data.max_text_tokens,
    sample_stride=config.data.sample_stride,
    max_samples=2,
    stats=None,
)

sample = dataset[0]
print('images.shape =', tuple(sample['images'].shape))
print('proprio.shape =', tuple(sample['proprio'].shape))
print('action.shape =', tuple(sample['action'].shape))
print('input_ids.shape =', tuple(sample['input_ids'].shape))

model = OpenVLAStylePolicy(
    vocab_size=tokenizer.vocab_size,
    proprio_dim=sample['proprio'].shape[-1],
    action_dim=sample['action'].shape[-1],
    num_cameras=sample['images'].shape[0],
    hidden_dim=config.model.hidden_dim,
    num_heads=config.model.num_heads,
    depth=config.model.depth,
    mlp_ratio=config.model.mlp_ratio,
    dropout=config.model.dropout,
)

with torch.no_grad():
    pred = model(
        images=sample['images'].unsqueeze(0),
        proprio=sample['proprio'].unsqueeze(0),
        input_ids=sample['input_ids'].unsqueeze(0),
        attention_mask=sample['attention_mask'].unsqueeze(0),
    )

print('pred.shape =', tuple(pred.shape))

## 3. 怎么正式训练

你可以先用 smoke 配置确认环境，再换到正式配置。

smoke:

```bash
cd /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/openvla_line
python scripts/smoke_train.py
```

正式训练:

```bash
cd /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/openvla_line
python scripts/train_openvla.py \
  --config configs/libero_spatial_small.json \
  --device cuda
```

如果你后面把这个目录单独搬走，只需要保证：

- `requirements.txt` 依赖装齐
- `--data-root` 指到新的 `LIBERO` 数据目录
- 再跑同样的脚本即可


In [ ]:
summary_path = RUN_ROOT / 'openvla_libero_spatial_smoke' / 'training_summary.json'
if summary_path.exists():
    print(summary_path)
    print(summary_path.read_text())
else:
    print('还没有 smoke 训练结果，先运行 scripts/smoke_train.py')